# MMCCP v2.0 — Google Colab Runner (`UNTESTED`)

> **UNTESTED on Colab.** Written from the code and validated only statically (JSON, syntax, URL/size/SHA parity). If a cell fails, see ['If something fails'](#if-something-fails) below or [`docs/run/03_GOOGLE_COLAB.md`](https://github.com/ShamantN/CAPSTONE/blob/main/docs/run/03_GOOGLE_COLAB.md).
>
> **Storage & Hardware Target:** Select `Runtime -> Change runtime type -> T4 GPU` (16 GB VRAM) and attach a Google Drive with **>= 30 GB free space** so WebUI Forge & ~14.2–25 GB of diffusion/swap weights survive session disconnects. Note: `open-in-Colab` from `main` resolves after PR #35 merges.
>
> **Safety Rules:** Uses **DEV** seed `11` only (`ebright.json`). Never run `TEST-2` or `RESERVED` LoRA seeds.

## Cell 0 — Check GPU, Mount Google Drive, Load Secrets & Export Environment Paths

This mounts `/content/drive`, creates `/content/drive/MyDrive/MMCCP/` cache folders, checks free Drive capacity (`>= 25 GB` recommended for first download), and safely reads optional `HF_TOKEN` / `NGROK_TOKEN` from Colab Secrets (`userdata`).

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# 1. Check attached GPU
print("=== GPU Check ===")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

# 2. Mount Google Drive
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    userdata = None
    print("[WARN] google.colab not detected (dry-run / non-Colab kernel).")

# 3. Persistent workspace paths
DRIVE_ROOT = Path("/content/drive/MyDrive/MMCCP") if Path("/content/drive").exists() else Path("/content/MMCCP_persist")
REPO_DIR = Path("/content/CAPSTONE")
FORGE_DIR = DRIVE_ROOT / "stable-diffusion-webui-forge"
MODELS_DIR = FORGE_DIR / "models"
HF_CACHE = DRIVE_ROOT / "hf_cache"
DEEPFACE_HOME = DRIVE_ROOT / "deepface_cache"
OUTPUT_ARCHIVE_DIR = DRIVE_ROOT / "outputs_archive"

for d in [DRIVE_ROOT, HF_CACHE, DEEPFACE_HOME, OUTPUT_ARCHIVE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

usage = shutil.disk_usage(str(DRIVE_ROOT))
print(f"Drive root: {DRIVE_ROOT} | Free space: {usage.free / (1024**3):.2f} GiB")
if usage.free / (1024**3) < 25.0:
    print("[WARN] Less than 25 GiB free — minimum weights + Forge take ~22–25 GiB.")

# 4. Read Colab Secrets safely (never echo token strings)
def _load_secret(name: str):
    if userdata is None:
        return None
    try:
        val = userdata.get(name)
        return val.strip() if val else None
    except Exception:
        return None

hf_tok = _load_secret("HF_TOKEN")
ngrok_tok = _load_secret("NGROK_TOKEN")
if hf_tok:
    os.environ["HF_TOKEN"] = hf_tok
if ngrok_tok:
    os.environ["NGROK_TOKEN"] = ngrok_tok

print(f"HF_TOKEN: {'SET' if os.environ.get('HF_TOKEN') else 'UNSET (optional, helps avoid HF throttling)'}")
print(f"NGROK_TOKEN: {'SET' if os.environ.get('NGROK_TOKEN') else 'UNSET (optional; cloudflared needs no token)'}")

# 5. Export MMCCP portability environment variables
os.environ["PYTHONUTF8"] = "1"
os.environ["MMCCP_ROOT"] = str(REPO_DIR)
os.environ["FORGE_DIR"] = str(FORGE_DIR)
os.environ["FORGE_MODELS_DIR"] = str(MODELS_DIR)
os.environ["FORGE_URL"] = "http://127.0.0.1:7860"
os.environ["FORGE_PYTHON"] = str(Path(subprocess.check_output(["which", "python3"], text=True).strip()))
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["DEEPFACE_HOME"] = str(DEEPFACE_HOME)
print("Environment configured cleanly.")

## Cell 1 — Clone MMCCP Repository & Verify Frozen Verifier Integrity

Clones `ShamantN/CAPSTONE` into `/content/CAPSTONE` (or pulls updates if already present) and checks `tools/check_freeze.py` (`SHA: 3fa69071f888643b...`). Set `GIT_BRANCH` to `main` after PR merge, or keep `docs/run-guides` while testing pre-merge.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ShamantN/CAPSTONE.git"
GIT_BRANCH = os.environ.get("MMCCP_BRANCH", "docs/run-guides")
REPO_DIR = Path(os.environ["MMCCP_ROOT"])

if not (REPO_DIR / ".git").exists():
    !git clone --branch {GIT_BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch --prune origin
    !git -C {REPO_DIR} checkout {GIT_BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {GIT_BRANCH}

%cd /content/CAPSTONE
!git --no-pager log -1 --oneline
!python3 tools/check_freeze.py
!python3 tools/audit_test2_isolation.py

## Cell 2 — Install Linux System Libraries (`aria2`, `g++`, OpenCV deps)

Installs `aria2` for resumable multi-connection HuggingFace/GitHub downloads plus build tools (`build-essential`, `g++`, `python3-dev`) in case `insightface` compiles native extensions for Colab's current Python version.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq aria2 build-essential g++ python3-dev libgl1 libglib2.0-0 ffmpeg
!aria2c --version | head -n 2

## Cell 3 — Install Python Dependencies Around Colab's GPU PyTorch

> **Colab Version Rule:** Do **NOT** blindly downgrade Colab's PyTorch/CUDA build unless your Colab VM already runs Python 3.10 + CUDA 12.1. Here we inspect `torch` first, keep Colab's GPU `torch`, try installing `requirements.txt` + `insightface` + `onnxruntime-gpu`, and pin `numpy>=1.26.0,<2.0.0` where compatible.

In [ ]:
import sys, torch
print("Python:", sys.version.split()[0], "| Colab Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA GPU runtime required! Change runtime type -> T4 GPU."

# Install MMCCP dependencies (keep Colab's GPU torch; install insightface + onnxruntime-gpu)
!pip install -q "numpy>=1.26.0,<2.0.0"
!pip install -q -r /content/CAPSTONE/requirements.txt
!pip install -q insightface==0.7.3 onnxruntime-gpu

# Sanity check core imports
!python3 -c "import torch, numpy, cv2, deepface, insightface, transformers, ultralytics, streamlit; print('Imports OK | torch:', torch.__version__, 'numpy:', numpy.__version__)" 

## Cell 4 — Setup WebUI Forge & Required 4 Extensions on Google Drive

Clones `lllyasviel/stable-diffusion-webui-forge` and the exact 4 extensions (`adetailer`, `sd-forge-couple`, `sd-webui-controlnet`, `sd-webui-reactor-sfw`) onto persistent Google Drive (skipped automatically if already installed). Also creates an idempotent Drive link so Forge finds weights immediately.

In [ ]:
import os
from pathlib import Path

FORGE_DIR = Path(os.environ["FORGE_DIR"])
if not (FORGE_DIR / ".git").exists():
    !git clone https://github.com/lllyasviel/stable-diffusion-webui-forge.git "{FORGE_DIR}"
else:
    print("Forge already present on Drive:", FORGE_DIR)

# Pin tested upstream Forge commit so recommended MMCCP patches apply cleanly
!git -C "{FORGE_DIR}" checkout dfdcbab685e57677014f05a3309b48cc87383167

ext_dir = FORGE_DIR / "extensions"
ext_dir.mkdir(parents=True, exist_ok=True)

EXTENSIONS = {
    "adetailer": "https://github.com/Bing-su/adetailer.git",
    "sd-forge-couple": "https://github.com/Haoming02/sd-forge-couple.git",
    "sd-webui-controlnet": "https://github.com/Mikubill/sd-webui-controlnet.git",
    "sd-webui-reactor": "https://github.com/Gourieff/sd-webui-reactor-sfw.git",
}

for folder, url in EXTENSIONS.items():
    target = ext_dir / folder
    if not target.exists():
        !git clone "{url}" "{target}"
    else:
        print(f"[OK] Extension already present: {folder}")

## Cell 5 — Apply Recommended MMCCP Forge Patches

Runs `tools/apply_forge_patches.py --apply` inside Drive's Forge clone (`dfdcbab685e57677014f05a3309b48cc87383167`) to apply the `3` recommended hardening patches (`IPAdapterPlus.py`, `controlnet.py`, `modules_forge/utils.py`). Safe and idempotent.

In [ ]:
%cd /content/CAPSTONE
!python3 tools/apply_forge_patches.py --forge-dir "$FORGE_DIR" --apply
!python3 tools/apply_forge_patches.py --forge-dir "$FORGE_DIR" --check

## Cell 6 — Download Required Model Weights & Verify Exact Sizes / SHA-256

Downloads the required `v2 default` weights directly to Drive using resumable `aria2c -c` (~14.2 GiB total core weights). Crucially saves `OpenPoseXL2.safetensors` under the exact filename **`openpose_xl.safetensors`** so Forge exposes `openpose_xl [f4251cb4]`.

*(Skipped automatically for any file that already exists with matching byte count!)*

In [ ]:
import hashlib
import os
import subprocess
from pathlib import Path

MODELS_DIR = Path(os.environ["FORGE_MODELS_DIR"])
REPO_DIR = Path(os.environ["MMCCP_ROOT"])

REQUIRED_DOWNLOADS = [
    {
        "name": "Juggernaut-XL v9",
        "url": "https://huggingface.co/RunDiffusion/Juggernaut-XL-v9/resolve/main/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors",
        "dest": MODELS_DIR / "Stable-diffusion" / "Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors",
        "size": 7105348188,
        "sha256": "c9e3e68f89b8e38689e1097d4be4573cf308de4e3fd044c64ca697bdb4aa8bca",
    },
    {
        "name": "ControlNet OpenPose XL (saved as openpose_xl.safetensors)",
        "url": "https://huggingface.co/thibaud/controlnet-openpose-sdxl-1.0/resolve/main/OpenPoseXL2.safetensors",
        "dest": MODELS_DIR / "ControlNet" / "openpose_xl.safetensors",
        "size": 5004167829,
        "sha256": "5a4b928cb1e93748217900cb66d4135bf70d932d2924232f925910fad9e43a92",
    },
    {
        "name": "IP-Adapter Plus-Face SDXL",
        "url": "https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter-plus-face_sdxl_vit-h.safetensors",
        "dest": MODELS_DIR / "ControlNet" / "ip-adapter-plus-face_sdxl_vit-h.safetensors",
        "size": 847517512,
        "sha256": "677ad8860204f7d0bfba12d29e6c31ded9beefdf3e4bbd102518357d31a292c1",
    },
    {
        "name": "CLIP-ViT-H-14 Image Encoder",
        "url": "https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors",
        "dest": MODELS_DIR / "ControlNetPreprocessor" / "CLIP-ViT-H-14.safetensors",
        "size": 2528373448,
        "sha256": "6ca9667da1ca9e0b0f75e46bb030f7e011f44f86cbfb8d5a36590fcd7507b030",
    },
    {
        "name": "InsightFace Swapper (inswapper_128.onnx)",
        "url": "https://huggingface.co/datasets/Gourieff/ReActor/resolve/main/models/inswapper_128.onnx",
        "dest": MODELS_DIR / "insightface" / "inswapper_128.onnx",
        "size": 554253681,
        "sha256": "e4a3f08c753cb72d04e10aa0f7dbe3deebbf39567d4ead6dce08e98aa49e16af",
    },
    {
        "name": "CodeFormer Restoration",
        "url": "https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth",
        "dest": MODELS_DIR / "Codeformer" / "codeformer-v0.1.0.pth",
        "size": 376637898,
        "sha256": "1009e537e0c2a07d4cabce6355f53cb66767cd4b4297ec7a4a64ca4b8a5684b7",
    },
    {
        "name": "YOLOv8n Verifier Person Detector",
        "url": "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt",
        "dest": REPO_DIR / "yolov8n.pt",
        "size": 6549796,
        "sha256": "f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36",
    },
]

hf_tok = os.environ.get("HF_TOKEN")
for item in REQUIRED_DOWNLOADS:
    dest: Path = item["dest"]
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size == item["size"]:
        print(f"[SKIP] {item['name']} already verified on disk ({item['size']:,} B)")
        continue
    print(f"[DL] {item['name']} -> {dest}")
    cmd = ["aria2c", "-c", "-x", "8", "-s", "8", "-k", "1M", "-d", str(dest.parent), "-o", dest.name]
    if hf_tok and "huggingface.co" in item["url"]:
        cmd += [f"--header=Authorization: Bearer {hf_tok}"]
    cmd.append(item["url"])
    subprocess.run(cmd, check=True)
    assert dest.stat().st_size == item["size"], f"Size mismatch on {dest}: {dest.stat().st_size} != {item['size']}"
    print(f"[OK] Downloaded {item['name']} ({dest.stat().st_size:,} B)")

## Cell 7 — Pre-Forge Preflight Verification

Checks environment paths, Python dependencies, GPU VRAM (`~15.3 GiB` on T4), SHA-256 checksums of downloaded weights, and frozen verifier hash before starting the Forge server.

In [ ]:
%cd /content/CAPSTONE
!python3 tools/verify_setup.py --mode colab --skip-forge --verify-sha256 --fix-hints

## Cell 8 — Launch WebUI Forge in Background & Wait Until API Ready

Starts Forge cleanly with `--api --xformers --port 7860`, logging into `/content/forge.log`. Polls `http://127.0.0.1:7860/sdapi/v1/sd-models` up to 480 seconds (first Forge boot installs extension packages like `insightface` / `ultralytics` inside Forge); automatically prints the last 50 log lines if it times out or exits early.

In [ ]:
import os
import subprocess
import time
import urllib.request
from pathlib import Path

FORGE_DIR = Path(os.environ["FORGE_DIR"])
LOG_PATH = Path("/content/forge.log")

# Kill any stale Forge from prior cell retry
!pkill -f "launch.py" || true

# Configure Forge command flags
forge_cmd = [
    "python3",
    str(FORGE_DIR / "launch.py"),
    "--api",
    "--xformers",
    "--listen",
    "--port", "7860",
    "--skip-python-version-check",
]

log_f = open(LOG_PATH, "w", encoding="utf-8")
forge_proc = subprocess.Popen(
    forge_cmd,
    cwd=str(FORGE_DIR),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)
print(f"Started Forge PID={forge_proc.pid}, polling http://127.0.0.1:7860 ...")

ready = False
deadline = time.time() + 480
while time.time() < deadline:
    if forge_proc.poll() is not None:
        print(f"[ERROR] Forge process exited prematurely with code {forge_proc.returncode}!")
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:7860/sdapi/v1/sd-models", timeout=3) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(5)

if not ready:
    lines = LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()[-50:]
    print("=== Last 50 lines of /content/forge.log ===")
    print("\n".join(lines))
    raise RuntimeError("Forge API did not become ready within timeout.")

print("Forge API is READY at http://127.0.0.1:7860!")
!python3 -c "import requests; print('SD Models:', [m['title'] for m in requests.get('http://127.0.0.1:7860/sdapi/v1/sd-models').json()])"
!python3 -c "import requests; cn=requests.get('http://127.0.0.1:7860/controlnet/model_list').json().get('model_list', []); print('ControlNets:', [c for c in cn if 'openpose_xl' in c or 'ip-adapter-plus-face' in c])" 

## Cell 9 — Full Post-Forge Preflight & Core Unit Tests

Verifies live Forge connectivity, ControlNet availability, portability helpers, and split guardrails.

In [ ]:
%cd /content/CAPSTONE
!python3 tools/verify_setup.py --mode colab --fix-hints
!python3 -m unittest -v tests.test_portability tests.test_verify_setup tests.test_splits

## Cell 10 — 1-Scene DEV Smoke Test (`ebright.json`, Scene 1, Seed 11, `--mode fast`)

Runs a complete end-to-end **zero-cache** pipeline smoke test generating character anchors first and rendering Scene 1 (`--mode fast` skips M6 Detailer for faster smoke turnaround).

In [ ]:
%cd /content/CAPSTONE
!python3 run_from_scratch.py --pipeline v2 --mode fast --story storyboards/ebright.json --scene 1 --seed 11

from pathlib import Path
smoke_png = Path("/content/CAPSTONE/edu_outputs/v2_fast/scene_1_seed_11.png")
assert smoke_png.exists(), f"Missing smoke output: {smoke_png}"
print("Smoke output saved:", smoke_png, f"({smoke_png.stat().st_size:,} B)")

## Cell 11 — Full 6-Scene `v2 default` Zero-Cache Storyboard Demo

Generates all 5 fresh character anchors + all 6 Ebright storyboard scenes with **M1–M7 & M10** active (`seed 11`, DEV split). Expect ~15–22 minutes depending on GPU tier (T4 vs L4/A100).

In [ ]:
%cd /content/CAPSTONE
!python3 run_from_scratch.py --pipeline v2 --mode default --story storyboards/ebright.json --seed 11

## Cell 12 — Inspect Character Anchors & Scenes Inline + Archive Outputs to Google Drive

Displays freshly generated anchor portraits (`experiments/fresh_run_from_scratch/anchors/`) and storyboard scenes (`edu_outputs/v2_default/`) inline, verifies no unexpected dirty repo edits exist, and zips the entire output bundle safely to Google Drive.

In [ ]:
import os
import shutil
from datetime import datetime
from pathlib import Path
from IPython.display import display, Image as IPyImage

REPO_DIR = Path(os.environ["MMCCP_ROOT"])
ANCHORS_DIR = REPO_DIR / "experiments" / "fresh_run_from_scratch" / "anchors"
SCENES_DIR = REPO_DIR / "edu_outputs" / "v2_default"
ARCHIVE_DIR = Path("/content/drive/MyDrive/MMCCP/outputs_archive") if Path("/content/drive").exists() else Path("/content/MMCCP_persist/outputs_archive")
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

print("=== Fresh Character Anchors ===")
for p in sorted(ANCHORS_DIR.glob("*.png")):
    print("Anchor:", p.name)
    display(IPyImage(filename=str(p), width=256))

print("=== Rendered 6-Scene Storyboard (v2 default) ===")
for p in sorted(SCENES_DIR.glob("scene_*_seed_*.png")):
    print("Scene:", p.name)
    display(IPyImage(filename=str(p), width=480))

# Zip outputs to Drive
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
bundle_dir = Path(f"/content/mmccp_colab_bundle_{ts}")
bundle_dir.mkdir(parents=True, exist_ok=True)
for sub, dst_name in [(SCENES_DIR, "v2_default"), (ANCHORS_DIR, "anchors")]:
    if sub.exists():
        shutil.copytree(sub, bundle_dir / dst_name, dirs_exist_ok=True)
if (REPO_DIR / "hardcoded_cache.json").exists():
    shutil.copy2(REPO_DIR / "hardcoded_cache.json", bundle_dir / "hardcoded_cache.json")

zip_path = shutil.make_archive(str(ARCHIVE_DIR / f"mmccp_colab_run_{ts}"), "zip", root_dir=bundle_dir)
print("Saved archive to Drive:", zip_path)

## Cell 13 — Optional: Launch Streamlit Interactive UI via `cloudflared` Tunnel

Opens the 4-tab Streamlit UI (`app.py`) over a temporary Cloudflare quick-tunnel URL (no account required). Stop this cell whenever you finish browsing.

In [ ]:
import re, subprocess, time

# Download cloudflared binary cleanly if absent
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

!pkill -f "streamlit run" || true
st_proc = subprocess.Popen(
    ["python3", "-m", "streamlit", "run", "/content/CAPSTONE/app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(4)

cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print("Waiting for trycloudflare.com URL (interrupt cell when done browsing)...")
for line in iter(cf_proc.stdout.readline, ""):
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("Open Streamlit Studio UI:", m.group(0))
        break

## Cell 14 — Clean Shutdown & Final Checks

Terminates Forge & Streamlit cleanly, verifies frozen verifier integrity (`tools/check_freeze.py`), and checks `git status`.

In [ ]:
!pkill -f "launch.py" || true
!pkill -f "streamlit run" || true
!pkill -f "cloudflared" || true
%cd /content/CAPSTONE
!python3 tools/check_freeze.py
!git --no-pager status -s

## Cell 15 — Drive Reconnect / Resume Cell (Use After Colab Session Disconnect!)

If your Colab runtime disconnected after idle timeout, run **ONLY Cell 0 -> Cell 2 -> Cell 3 -> THIS Cell 15** instead of Cells 4–6: all ~15–22 GB of Forge code and weights are already on Google Drive!

In [ ]:
import os
import subprocess
import time
import urllib.request
from pathlib import Path

assert Path(os.environ["FORGE_MODELS_DIR"]).exists(), "Mount Drive cleanly in Cell 0 first!"
REPO_DIR = Path(os.environ["MMCCP_ROOT"])
FORGE_DIR = Path(os.environ["FORGE_DIR"])

if not (REPO_DIR / ".git").exists():
    !git clone --branch {os.environ.get("MMCCP_BRANCH", "docs/run-guides")} https://github.com/ShamantN/CAPSTONE.git "{REPO_DIR}"

!pkill -f "launch.py" || true
log_f = open("/content/forge.log", "w", encoding="utf-8")
proc = subprocess.Popen(
    ["python3", str(FORGE_DIR / "launch.py"), "--api", "--xformers", "--listen", "--port", "7860", "--skip-python-version-check"],
    cwd=str(FORGE_DIR),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)
for _ in range(96):
    try:
        with urllib.request.urlopen("http://127.0.0.1:7860/sdapi/v1/sd-models", timeout=3) as r:
            if r.status == 200:
                print("Forge READY on reconnect!")
                break
    except Exception:
        time.sleep(5)

## If something fails *(Top 8 Fixes — All Unverified on Colab)*

1. **`CUDA GPU runtime required` (Cell 3) `[unverified on Colab]`:** Switch `Runtime -> Change runtime type -> T4 GPU`, then rerun from Cell 0.
2. **Repo clone fails or GitHub URL 404 `[unverified on Colab]`:** PR #35 not yet merged (`MMCCP_BRANCH="docs/run-guides"`) or using a private fork (add `GITHUB_TOKEN` in Secrets).
3. **`insightface==0.7.3` fails compiling (Cell 3) `[unverified on Colab]`:** Ensure Cell 2 (`build-essential g++ python3-dev`) finished and pass `--no-build-isolation`.
4. **NumPy upgraded to `2.x` `[unverified on Colab]`:** Re-run `!pip install "numpy>=1.26.0,<2.0.0" --force-reinstall --no-deps` and restart runtime.
5. **HuggingFace `429` throttle (Cell 6) `[unverified on Colab]`:** Add `HF_TOKEN` in Colab Secrets -> rerun Cell 0 -> rerun Cell 6 (`aria2c -c` resumes).
6. **Forge startup timeout (Cell 8) `[unverified on Colab]`:** Inspect the printed tail of `/content/forge.log`, fix any missing wheel, and rerun Cell 8.
7. **`RuntimeError: cpu and cuda:0` in `IPAdapterPlus.py` `[unverified on Colab]`:** Re-run Cell 4 (`git checkout dfdcbab685e57677014f05a3309b48cc87383167`) + Cell 5 (`apply_forge_patches.py --apply`).
8. **`cloudflared` tunnel fails (Cell 13) `[unverified on Colab]`:** Switch to `NGROK_TOKEN` (`pyngrok`) or inspect images inline / via `MyDrive/MMCCP/outputs_archive/`.